## CSCI-4041 - Algorithms - Midterm Project

## Part 2: AVL Tree Implementation

### 3/2026

**Sources:**
- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). *Introduction to Algorithms*, 4th ed. MIT Press.
- Upton Azzam, J. (2026). *CSCI-4041 Ch13 Balanced Search Trees* [Course notebook].

<mark>_____________________________________________________________________________________</mark>

## Load the Binary Search Tree and Balanced Tree

In [ ]:
%%capture
%run Ch12_BinarySearchTree.ipynb
%run Ch13_BalancedSearchTrees.ipynb

<mark>_____________________________________________________________________________________</mark>

## AVL Tree

The following class ```avltree``` is a child of ```balancedtree``` and overrides the ```insert``` and ```delete``` methods to enforce the AVL balance invariant. The invariant requires that the balance factor ```bf = left_height - right_height``` satisfies ```|bf| <= 1``` at every node after every operation. When a violation is detected during the walk-up, one of four rotation cases (LL, LR, RR, RL) restores balance.

The insert method calls ```super().insert(k)``` to place the node using the Ch13 logic, then walks up via parent pointers applying at most one rotation. The delete method follows the Ch13 template structure: iterative search, ```transplant()``` for removal, then a full walk-up to the root since delete may require rebalancing at multiple ancestors.

In [ ]:
class avltree(balancedtree):
    """AVL tree: child of balancedtree that enforces the AVL balance invariant"""

    ##################################################################### insert

    def insert(self, k):
        """insert key k and rebalance -- at most one rotation needed"""
        binarysearchtree.insert(self, k)        # raw BST insert only -- skip balancedtree walk-up
                                                # so our walk-up below handles heights + balance together

        x = self.search(self.root, k)           # find the inserted node
        if x is None or x.key != k:
            return

        while x.p:                              # walk up: update heights and fix balance
            x = x.p
            lh, rh = self.update_height(x)      # recompute height, get lh and rh
            bf = lh - rh                        # balance factor at x

            if bf > 1:                                        # left heavy
                lhl, rhl = self.compute_height(x.left)
                saved = x.p                               # save parent before rotation moves x down
                if lhl - rhl >= 0:                            # LL case
                    self.right_rotate(x)
                else:                                         # LR case
                    self.left_rotate(x.left)
                    self.right_rotate(x)
                if saved:
                    self.update_height(saved)
                break                                         # insert needs at most one fix

            elif bf < -1:                                     # right heavy
                lhr, rhr = self.compute_height(x.right)
                saved = x.p                               # save parent before rotation moves x down
                if rhr - lhr >= 0:                            # RR case
                    self.left_rotate(x)
                else:                                         # RL case
                    self.right_rotate(x.right)
                    self.left_rotate(x)
                if saved:
                    self.update_height(saved)
                break                                         # insert needs at most one fix

    ##################################################################### delete

    def delete(self, k):
        """delete key k and rebalance -- multiple rotations may be needed"""

        # Phase 1: find node z with key k
        x = self.root
        z = None
        while x != None:
            z = x
            if z.key == k:                      # found
                break
            elif k < x.key:
                x = x.left
            else:
                x = x.right

        if z is None or z.key != k:             # key not in tree
            return

        print("key =", k, "was found")
        save_parent = z.p                       # save for walk-up starting point

        # Phase 2: remove node z (3 cases matching Ch13 template)
        if z.left == None:                      # Case A: no left child
            self.transplant(z, z.right)

        elif z.right == None:                   # Case B: no right child
            self.transplant(z, z.left)

        else:                                   # Case C: two children
            y = self.min(z.right)               # in-order successor
            save_parent = y.p                   # walk-up starts from successor's old position
            if y != z.right:
                self.transplant(y, y.right)     # y can only have a right child
                y.right = z.right
                y.right.p = y
            self.transplant(z, y)
            y.left = z.left
            y.left.p = y
            self.update_height(y)               # fix y's height after relinking
            if save_parent == z:                # y was direct right child -- walk up from y
                save_parent = y

        # Phase 3: walk up to root rebalancing wherever needed
        x = save_parent
        while x != None:
            lh, rh = self.update_height(x)      # recompute height
            bf = lh - rh                        # balance factor at x

            if bf > 1:                                        # left heavy
                lhl, rhl = self.compute_height(x.left)
                next_x = x.p                              # save before rotation moves x down
                if lhl - rhl >= 0:                            # LL case
                    self.right_rotate(x)
                else:                                         # LR case
                    self.left_rotate(x.left)
                    self.right_rotate(x)
                x = next_x
                continue

            elif bf < -1:                                     # right heavy
                lhr, rhr = self.compute_height(x.right)
                next_x = x.p                              # save before rotation moves x down
                if rhr - lhr >= 0:                            # RR case
                    self.left_rotate(x)
                else:                                         # RL case
                    self.right_rotate(x.right)
                    self.left_rotate(x)
                x = next_x
                continue

            x = x.p                             # continue up -- no break, delete may need multiple rotations

<mark>_____________________________________________________________________________________</mark>

## Example Usage

In [ ]:
AVL = avltree()

AVL.insert(4)
AVL.insert(2)
AVL.insert(5)
AVL.insert(1)
AVL.insert(3)

PrintTree(AVL)

In [ ]:
# LL case -- inserting in descending order forces a right rotation
AVL2 = avltree()
for k in [30, 20, 10]:
    AVL2.insert(k)
PrintTree(AVL2)

In [ ]:
# RR case -- inserting in ascending order forces a left rotation
AVL3 = avltree()
for k in [10, 20, 30]:
    AVL3.insert(k)
PrintTree(AVL3)

In [ ]:
# LR case -- forces left then right rotation
AVL4 = avltree()
for k in [30, 10, 20]:
    AVL4.insert(k)
PrintTree(AVL4)

In [ ]:
# RL case -- forces right then left rotation
AVL5 = avltree()
for k in [10, 30, 20]:
    AVL5.insert(k)
PrintTree(AVL5)

In [ ]:
# deletion case A -- leaf node
AVL.delete(1)
PrintTree(AVL)

In [ ]:
# deletion case B -- node with one child
AVL.delete(2)
PrintTree(AVL)

In [ ]:
# deletion case C -- node with two children
AVL.delete(4)
PrintTree(AVL)

<mark>_____________________________________________________________________________________</mark>

## Correctness Check

The following function verifies that a given tree satisfies both the BST property and the AVL invariant at every node.

In [ ]:
def verify_avl(T):
    """checks BST ordering and AVL balance invariant at every node"""
    def check(node, lo, hi):
        if node is None:
            return True
        if not (lo < node.key < hi):
            print("BST violation at node", node.key)
            return False
        lh, rh = T.compute_height(node)
        if abs(lh - rh) > 1:
            print("AVL violation at node", node.key, "bf =", lh - rh)
            return False
        return check(node.left, lo, node.key) and check(node.right, node.key, hi)
    return check(T.root, float('-inf'), float('inf'))

In [ ]:
import random

# build a tree from random keys and verify after every insert
random.seed(42)
keys = random.sample(range(1, 200), 30)

T = avltree()
for k in keys:
    T.insert(k)
    assert verify_avl(T), f"AVL violated after inserting {k}"

print("all inserts valid")
PrintTree(T)

In [ ]:
# verify after every delete
for k in keys[:15]:
    T.delete(k)
    assert verify_avl(T), f"AVL violated after deleting {k}"

print("all deletes valid")
PrintTree(T)